# Taller 05: Clustering DBSCAN

Generamos datos sintéticos 2D a partir de una figura con grupos amorfos dibujada manualmente, agregamos outliers uniformes, y aplicamos DBSCAN, seleccionando `eps` y `MinPts` mediante análisis de k-distancia y evaluando el resultado con dos métricas de `sklearn.metrics`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score

np.random.seed(42)

## 1. Figura original
Imagen dibujada manualmente (mínimo 20 grupos amorfos), fondo blanco.

In [ ]:
img = Image.open("figura_clusters.png").convert("L")
ancho, alto = img.size
arr = np.array(img)

plt.imshow(arr, cmap="gray")
plt.title("Figura original")
plt.axis("off")
plt.show()

## 2. Extracción de coordenadas y generación de datos sintéticos
Se detectan los píxeles pintados, se muestrean y se les agrega ruido gaussiano para formar el dataset 2D.

In [ ]:
mascara = arr < 240
filas, cols = np.where(mascara)
coords = np.column_stack([cols, alto - filas]).astype(float)

n_muestras = 1500
idx = np.random.choice(len(coords), n_muestras, replace=len(coords) < n_muestras)
datos = coords[idx] + np.random.normal(0, 1.5, size=(n_muestras, 2))

plt.scatter(datos[:,0], datos[:,1], s=4)
plt.title("Datos sintéticos generados a partir de la figura")
plt.show()

## 3. Outliers uniformes

In [ ]:
n_outliers = int(0.08 * n_muestras)
x_min, y_min = datos.min(axis=0)
x_max, y_max = datos.max(axis=0)
outliers = np.random.uniform([x_min, y_min], [x_max, y_max], size=(n_outliers, 2))

dataset = np.vstack([datos, outliers])

plt.scatter(datos[:,0], datos[:,1], s=4, label="Figura")
plt.scatter(outliers[:,0], outliers[:,1], s=10, c="red", marker="x", label="Outliers")
plt.legend()
plt.title("Dataset completo")
plt.show()

## 4. Análisis de k-distancia
Buscamos el punto de inflexión para justificar `eps`, usando `k = MinPts` candidato.

In [ ]:
k = 10  # MinPts candidato
vecinos = NearestNeighbors(n_neighbors=k).fit(dataset)
distancias, _ = vecinos.kneighbors(dataset)
k_dist = np.sort(distancias[:, -1])

plt.plot(k_dist)
plt.xlabel("Puntos ordenados")
plt.ylabel(f"Distancia al {k}-ésimo vecino")
plt.title("Gráfico de k-distancia")
plt.show()

El codo de la curva anterior se ubica aproximadamente en **eps ≈ [leer del gráfico]**, con `MinPts = 10`. Estos valores se usan a continuación para DBSCAN.

## 5. Aplicación de DBSCAN

In [ ]:
eps = 6
min_pts = 10

modelo = DBSCAN(eps=eps, min_samples=min_pts)
etiquetas = modelo.fit_predict(dataset)

n_clusters = len(set(etiquetas)) - (1 if -1 in etiquetas else 0)
n_ruido = np.sum(etiquetas == -1)
print(f"Clusters encontrados: {n_clusters} | Puntos de ruido: {n_ruido}")

## 6. Evaluación
Usamos **Silhouette Score** (más alto es mejor) y **Davies-Bouldin Index** (más bajo es mejor), excluyendo el ruido (`-1`) porque ninguna de las dos métricas está definida para tratarlo como un cluster válido.

In [ ]:
mask = etiquetas != -1
sil = silhouette_score(dataset[mask], etiquetas[mask])
db = davies_bouldin_score(dataset[mask], etiquetas[mask])
print(f"Silhouette Score: {sil:.3f}")
print(f"Davies-Bouldin Index: {db:.3f}")

## 7. Resultado final

In [ ]:
plt.figure(figsize=(8,6))
for c in set(etiquetas):
    puntos = dataset[etiquetas == c]
    if c == -1:
        plt.scatter(puntos[:,0], puntos[:,1], c="black", marker="x", s=20, label="Ruido")
    else:
        plt.scatter(puntos[:,0], puntos[:,1], s=8, label=f"Cluster {c}")

plt.title(f"DBSCAN | eps={eps}, MinPts={min_pts} | Clusters: {n_clusters}")
plt.xlabel("x"); plt.ylabel("y")
plt.show()

## 8. Conclusiones
DBSCAN encontró `n_clusters` clusters, identificando como ruido principalmente los outliers uniformes. Al ser un método basado en densidad, se ajusta mejor que K-Means a las formas amorfas de la figura dibujada, sin necesitar definir el número de clusters de antemano.